In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

In [ ]:
import os
DATA_PATH = os.environ.get(
    "MONETARY_DATA_PATH", os.path.join("..", "data", "Monetary_transmission_data_III.xlsx")
)
FEATURE_COLS = ["CPI", "EXR", "M2b", "RSV", "BRNT"]
TARGET_COLS = ["CPI", "EXR"]
TEST_FRAC = 0.15
MIN_TEST = 4
MAXLAGS = 6
SIG_LEVEL = 0.05  # Johansen trace test threshold for the cointegration-rank decision

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load raw Excel data and enforce a sorted monthly DatetimeIndex."""
    df = pd.read_excel(path, parse_dates=["DATE"])
    return df.sort_values("DATE").set_index("DATE").asfreq("MS")


df = load_data(DATA_PATH)[FEATURE_COLS]
df.head()

In [ ]:
# Structural-break dates partition the sample into four regimes; "W5" retains
# the entire series as a fifth, unsegmented baseline for comparison.
BREAK_DATES = pd.to_datetime(["2008-06-01", "2016-10-01", "2022-03-01"])
EDGES = [df.index.min(), *BREAK_DATES, df.index.max() + pd.DateOffset(days=1)]

WINDOWS = {f"W{i + 1}": (EDGES[i], EDGES[i + 1] - pd.DateOffset(days=1)) for i in range(len(EDGES) - 1)}
WINDOWS["W5"] = (df.index.min(), df.index.max())

for label, (start, end) in WINDOWS.items():
    print(f"{label}: {start.date()} -> {end.date()} ({len(df.loc[start:end])} months)")

In [ ]:
def select_lag_order(train: pd.DataFrame, maxlags: int) -> int:
    """AIC-optimal VAR lag order, capped so the model stays identified on short windows."""
    cap = max(1, min(maxlags, len(train) // (len(train.columns) + 1) - 1))
    try:
        order = VAR(train).select_order(cap).aic
    except Exception:
        order = 1
    return max(order, 1)


def cointegration_rank(train: pd.DataFrame, lag_order: int, sig_level: float = SIG_LEVEL) -> int:
    """Johansen trace-test rank at `sig_level`; falls back to 0 (no cointegration) if infeasible."""
    cvt_col = {0.10: 0, 0.05: 1, 0.01: 2}[sig_level]
    try:
        result = coint_johansen(train, det_order=0, k_ar_diff=lag_order)
        return int((result.lr1 > result.cvt[:, cvt_col]).sum())
    except (np.linalg.LinAlgError, ValueError):
        return 0


def fit_forecast_window(df_window: pd.DataFrame, target_cols: list, test_size: int, maxlags: int = MAXLAGS) -> tuple:
    """Pick VECM (cointegrated) or VAR-in-differences (not cointegrated) and forecast the test horizon."""
    train, test = df_window.iloc[:-test_size], df_window.iloc[-test_size:]
    lag_order = select_lag_order(train, maxlags)
    rank = cointegration_rank(train, lag_order)

    if rank > 0:
        fitted = VECM(train, k_ar_diff=lag_order, coint_rank=rank, deterministic="ci").fit()
        forecast = pd.DataFrame(fitted.predict(steps=test_size), index=test.index, columns=train.columns)
        spec = f"VECM(rank={rank}, k_ar_diff={lag_order})"
    else:
        train_diff = train.diff().dropna()
        fitted = VAR(train_diff).fit(lag_order)
        diff_forecast = fitted.forecast(train_diff.values[-lag_order:], steps=test_size)
        forecast = pd.DataFrame(diff_forecast, index=test.index, columns=train.columns).cumsum() + train.iloc[-1]
        spec = f"VAR(p={lag_order}, differenced)"

    return forecast[target_cols], test[target_cols], spec


def evaluate(forecast_df: pd.DataFrame, actual_df: pd.DataFrame, target_cols: list) -> pd.DataFrame:
    """RMSE and MAE per target series."""
    return pd.DataFrame({
        col: {
            "RMSE": np.sqrt(mean_squared_error(actual_df[col], forecast_df[col])),
            "MAE": mean_absolute_error(actual_df[col], forecast_df[col]),
        }
        for col in target_cols
    }).T

In [ ]:
def run_window_pipeline(df_window: pd.DataFrame) -> dict:
    """Select a model, forecast, and evaluate within one structural-break window."""
    test_size = max(int(len(df_window) * TEST_FRAC), MIN_TEST)
    forecast_df, actual_df, spec = fit_forecast_window(df_window, TARGET_COLS, test_size)
    metrics = evaluate(forecast_df, actual_df, TARGET_COLS)
    return {"df_window": df_window, "spec": spec, "forecast": forecast_df, "actual": actual_df, "metrics": metrics}


results = {}
for label, (start, end) in WINDOWS.items():
    df_w = df.loc[start:end]
    results[label] = run_window_pipeline(df_w)
    print(f"{label}: {results[label]['spec']} | n={len(df_w)}")
    display(results[label]["metrics"])

In [ ]:
# Model spec and held-out test error per window.
summary_df = pd.concat({label: r["metrics"] for label, r in results.items()}, names=["window", "target"])
spec_df = pd.Series({label: r["spec"] for label, r in results.items()}, name="spec")
display(spec_df.to_frame())
summary_df

In [ ]:
fig, axes = plt.subplots(len(TARGET_COLS), len(WINDOWS), figsize=(5 * len(WINDOWS), 4 * len(TARGET_COLS)), sharey="row")
for col_idx, target in enumerate(TARGET_COLS):
    for win_idx, (label, r) in enumerate(results.items()):
        ax = axes[col_idx, win_idx]
        ax.plot(r["actual"].index, r["actual"][target], label="Actual")
        ax.plot(r["actual"].index, r["forecast"][target], label="Predicted")
        ax.set_title(f"{target} | {label}\n{r['spec']}", fontsize=9)
        ax.tick_params(axis="x", labelrotation=45)
axes[0, 0].legend()
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("../results", exist_ok=True)
# Standardized long-format export consumed by the model-comparison pipeline.
export_rows = [
    {"window": label, "target": col, "RMSE": r["metrics"].loc[col, "RMSE"], "MAE": r["metrics"].loc[col, "MAE"]}
    for label, r in results.items() for col in TARGET_COLS
]
pd.DataFrame(export_rows).to_csv(f"../results/metrics_var_vecm.csv", index=False)